## Neon connection

In [1]:
from sqlalchemy import create_engine, text, inspect

import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

READONLY_DATABASE_URL = os.getenv('READONLY_DATABASE_URL')

engine = create_engine(READONLY_DATABASE_URL)


with engine.connect() as conn:
    cities_df = pd.read_sql(text("SELECT * FROM public.cities;"), conn)

cities_df.head(10)


,id,name,country,country_code,continent,latitude,longitude,cell_lat,cell_lon,population,is_capital,latitude_band,coast_distance_km,coastal,priority,download_order,era5_lat,era5_lon,era5_elevation
0,phoenix-us,Phoenix,United States of America,US,North America,33.5419,-112.0719,33.50,-112.00,3551000,0,subtropical_north,271,0,1,1,33.50,-112.00,366.0
1,antananarivo-mg,Antananarivo,Madagascar,MG,Africa,-18.9147,47.5147,-19.00,47.50,1697000,1,tropical_south,162,0,1,2,-19.00,47.50,1250.0
2,auckland-nz,Auckland,New Zealand,NZ,Oceania,-36.8481,174.7630,-36.75,174.75,1377200,0,temperate_south,0,1,1,3,-36.75,174.75,36.0
3,jinxi-cn,Jinxi,China,CN,Asia,40.7523,120.8280,40.75,120.75,2426000,0,temperate_north,11,1,1,4,40.75,120.75,18.0
4,montevideo-uy,Montevideo,Uruguay,UY,South America,-34.8561,-56.1730,-34.75,-56.25,1513000,1,subtropical_south,6,1,1,5,-34.75,-56.25,53.0
5,lisbon-pt,Lisbon,Portugal,PT,Europe,38.7247,-9.1468,38.75,-9.25,2812000,1,temperate_north,2,1,1,6,38.75,-9.25,53.0
6,cilacap-id,Cilacap,Indonesia,ID,Asia,-7.7188,109.0154,-7.75,109.00,1174964,0,tropical_south,2,1,1,7,-7.50,109.00,7.0
7,isfahan-ir,Isfahan,Iran,IR,Asia,32.7020,51.6981,32.75,51.75,1628000,0,subtropical_north,318,0,1,8,32.75,51.75,1568.0
8,caracas-ve,Caracas,Venezuela,VE,South America,10.5029,-66.9190,10.50,-67.00,2985000,1,tropical_north,12,1,1,9,10.50,-67.00,905.0
9,yaounde-cm,Yaoundé,Cameroon,CM,Africa,3.8686,11.5147,3.75,11.50,1611000,1,tropical_north,191,0,1,10,3.75,11.50,714.0


## Cities map

In [2]:
import plotly.express as px
import plotly.graph_objects as go

CONTINENT_COLORS = {
    "Africa": "#E69F00",
    "Asia": "#009E73",
    "Europe": "#0072B2",
    "North America": "#D55E00",
    "South America": "#CC79A7",
    "Oceania": "#56B4E9",
    "Oceanic islands": "#8C9BA8",
}


def build_cities_map(cities, dark=False, height=640):
    text = "#E5E8EC" if dark else "#2F3742"
    muted = "#98A1AE" if dark else "#8A94A3"
    surface = "#1C2027" if dark else "#FFFFFF"
    border = "#39424F" if dark else "#E6E9EE"
    land = "#232A33" if dark else "#F5F7F9"
    ocean = "#151A21" if dark else "#E8F2F4"
    coastline = "#46515F" if dark else "#C9D2DC"
    capital = "#E5E8EC" if dark else "#202536"

    cities = cities.copy()
    cities["climate_zone"] = (
        cities["latitude_band"].str.replace("_", " ", regex=False).str.title()
        + " · "
        + cities["coastal"].map({1: "Coastal", 0: "Inland"}).fillna("Unknown")
    )
    cities["capital_label"] = cities["is_capital"].map({1: " · Capital", 0: ""}).fillna("")
    cities["marker_size"] = cities["population"] ** 0.35

    size_max = 26
    sizeref = 2.0 * cities["marker_size"].max() / (size_max**2)

    fig = px.scatter_geo(
        cities,
        lat="latitude",
        lon="longitude",
        color="continent",
        size="population",
        size_max=size_max,
        hover_name="name",
        custom_data=[
            "country",
            "country_code",
            "population",
            "climate_zone",
            "coast_distance_km",
            "capital_label",
        ],
        category_orders={"continent": list(CONTINENT_COLORS)},
        color_discrete_map=CONTINENT_COLORS,
        projection="natural earth",
        fitbounds="locations",
    )

    hovertemplate = (
        "<b>%{hovertext}</b><br>"
        "<span style='color:" + muted + "'>%{customdata[1]}</span> · %{customdata[0]}%{customdata[5]}<br><br>"
        "<span style='color:" + muted + "'>Population</span>&nbsp;&nbsp;%{customdata[2]:,}<br>"
        "<span style='color:" + muted + "'>Climate</span>&nbsp;&nbsp;%{customdata[3]}<br>"
        "<span style='color:" + muted + "'>Coast</span>&nbsp;&nbsp;%{customdata[4]:,.0f} km"
        "<extra></extra>"
    )

    fig.update_traces(
        hovertemplate=hovertemplate,
        marker=dict(
            sizemode="area",
            sizeref=sizeref,
            sizemin=4,
            opacity=0.9,
            line=dict(width=0.8, color=surface),
        ),
        selector=dict(type="scattergeo"),
    )

    capitals = cities[cities["is_capital"] == 1]
    fig.add_trace(
        go.Scattergeo(
            lat=capitals["latitude"],
            lon=capitals["longitude"],
            mode="markers",
            marker=dict(
                symbol="star",
                size=9,
                color=capital,
                line=dict(width=0.8, color=surface),
            ),
            name="Capital city",
            hoverinfo="skip",
        )
    )

    fig.update_geos(
        projection_type="natural earth",
        showcoastlines=True,
        coastlinecolor=coastline,
        coastlinewidth=0.6,
        showland=True,
        landcolor=land,
        showocean=True,
        oceancolor=ocean,
        showlakes=True,
        lakecolor=ocean,
        showrivers=False,
        showcountries=True,
        countrycolor=border,
        countrywidth=0.5,
        showframe=False,
        bgcolor="rgba(0,0,0,0)",
    )

    fig.update_layout(
        height=height,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=8, r=8, t=96, b=76),
        font=dict(
            family="Inter, Segoe UI, Helvetica, Arial, sans-serif",
            size=12,
            color=text,
        ),
        title=dict(
            text="<b>Global city coverage</b>",
            x=0.005,
            y=0.97,
            xanchor="left",
            yanchor="top",
            font=dict(size=22, color=text),
        ),
        annotations=[
            dict(
                text=f"{len(cities):,} cities · {cities['continent'].nunique()} regions · ERA5 reanalysis grid cells",
                x=0.005,
                y=0.90,
                xref="paper",
                yref="paper",
                showarrow=False,
                xanchor="left",
                yanchor="top",
                font=dict(size=12, color=muted),
            ),
            dict(
                text="Source: Open-Meteo ERA5 · Marker area ∝ population",
                x=1.0,
                y=0.97,
                xref="paper",
                yref="paper",
                showarrow=False,
                xanchor="right",
                yanchor="top",
                font=dict(size=10, color=muted),
            ),
        ],
        legend=dict(
            title=dict(text="Legend", font=dict(size=11, color=muted)),
            orientation="h",
            yanchor="top",
            y=-0.06,
            xanchor="left",
            x=0.0,
            font=dict(size=11, color=text),
            bgcolor="rgba(0,0,0,0)",
            itemsizing="constant",
            itemwidth=30,
        ),
        hoverlabel=dict(
            bgcolor=surface,
            bordercolor=border,
            font=dict(
                family="Inter, Segoe UI, Helvetica, Arial, sans-serif",
                size=12,
                color=text,
            ),
            align="left",
        ),
        hovermode="closest",
    )

    return fig


fig = build_cities_map(cities_df, dark=False)
fig.show()


In [2]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM public.daily ORDER BY date DESC LIMIT 10;"))
    daily_data = result.fetchall()
    daily_df = pd.DataFrame(daily_data)
    print(daily_df)

           city_id        date  t_mean  t_max  t_min
0        lisbon-pt  2026-11-07    17.4   20.9   14.6
1       caracas-ve  2026-11-07    22.8   27.7   20.5
2      auckland-nz  2026-11-07    18.0   21.3   15.7
3    montevideo-uy  2026-11-07    15.0   16.9   11.7
4       cilacap-id  2026-11-07    26.7   29.3   24.7
5       isfahan-ir  2026-11-07    14.5   19.3   10.7
6       phoenix-us  2026-11-07    20.0   26.2   16.1
7  antananarivo-mg  2026-11-07    21.8   29.2   14.6
8         jinxi-cn  2026-11-07     1.0    3.4   -1.2
9       yaounde-cm  2026-11-07    24.4   29.4   21.2
